# Clase 11 — Drift y reentrenamiento

**Curso:** MLOps y Monitorización con Python · **Bloque B**
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/11_drift_reentrenamiento.md`

## Qué verás en esta clase

- La diferencia entre **data drift** (cambia la distribución de las variables de entrada) y **concept drift** (cambia la relación entre las variables y la etiqueta), con un ejemplo pequeño de cada uno.
- Cómo medir *drift* con dos pruebas estadísticas simples implementadas por ti: **Kolmogorov-Smirnov** (`scipy.stats.ks_2samp`) y el **índice de estabilidad poblacional** (PSI, calculado a mano con buckets fijos).
- Cómo confirmar esas mismas conclusiones con la librería **Evidently**, sobre un cambio REAL entre `churn.csv` y `churn_v2.csv` (no una simulación de palabra).
- Cómo reentrenar el modelo sobre el dato con *drift* y medir su impacto real sobre el test original -- incluido un impacto que la métrica global no muestra, pero un análisis por segmento sí.
- Cómo definir un **disparador de reentrenamiento** explícito, y por qué reentrenar con datos ya contaminados puede perpetuar el problema que causó el *drift*.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook. Ningún número de esta clase se mide con el reloj real: todo viene de datos con semilla fija.

## Prerequisitos

- De las clases 4, 5, 7 y 10: la configuración ganadora del modelo (regresión logística, `class_weight="balanced"`, `C=1,0`) y el mismo conjunto de prueba de `churn.csv`.
- De la clase 6: la idea de que una métrica agregada puede esconder un problema real en un subgrupo (aquí se repite con el segmento "plan premium").
- De la clase 10: la distinción entre métricas de sistema y de modelo -- esta clase retoma justo la pregunta que dejó abierta: qué hacer cuando la distribución de las predicciones SÍ cambia.
- Python básico y `pandas`. No hace falta haber usado Evidently antes: esta clase la introduce desde cero.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte (Evidently, scipy).

In [ ]:
import warnings

# Silenciamos avisos de compatibilidad fijos (no aportan nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("scipy", "scipy")
asegurar("matplotlib", "matplotlib")
asegurar("evidently", "evidently")
print("Librerías listas.")

---
## Celda 0 — Setup

In [ ]:
import os
import shutil
import json
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Silenciamos avisos fijos de compatibilidad (no aportan nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("evidently").setLevel(logging.ERROR)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerías y funciones auxiliares listas.")

---
## Carpeta de trabajo efímera (se limpia al principio y al final)

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero (también al empezar, por si quedó algo de una ejecución anterior)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## Cargar las dos versiones de los datos (solo lectura)

In [ ]:
RUTA_V1 = cargar_datos("churn.csv")
RUTA_V2 = cargar_datos("churn_v2.csv")
df_v1 = pd.read_csv(RUTA_V1)
df_v2 = pd.read_csv(RUTA_V2)

print("churn.csv (referencia):", df_v1.shape)
print("churn_v2.csv (nuevo):", df_v2.shape)
assert list(df_v1.columns) == list(df_v2.columns), "las dos versiones deben tener el mismo esquema de columnas"
print("Mismo esquema de columnas en ambos archivos:", list(df_v1.columns))

---
## 1. Data drift frente a concept drift

- **Data drift**: cambia P(X), la distribución de las variables de entrada. La relación P(Y|X) -- cómo esas variables predicen la etiqueta -- sigue siendo la misma.
- **Concept drift**: cambia P(Y|X). La relación entre las variables y la etiqueta cambia, aunque las variables de entrada tengan exactamente la misma distribución.

> **Nota:** ambos tipos de *drift* pueden ocurrir a la vez, y en la práctica suele ser difícil separarlos sin la etiqueta real. Los dos ejemplos siguientes son deliberadamente pequeños para que la diferencia se vea sin ambigüedad.

In [ ]:
# Ejemplo A -- DATA DRIFT: la distribución de entrada cambia, la relación con la etiqueta NO.
# Dos poblaciones pequeñas de 4 clientes, con la MISMA regla de etiquetado: abandona si tickets_soporte >= 3.
poblacion_antes = pd.DataFrame({"tickets_soporte": [0, 1, 1, 2], "abandono": [0, 0, 0, 0]})
poblacion_despues = pd.DataFrame({"tickets_soporte": [3, 4, 3, 5], "abandono": [1, 1, 1, 1]})

regla_es_la_misma = True  # "abandona si tickets_soporte >= 3" no cambió: solo cambió QUIÉN aparece en los datos
for df_pob in (poblacion_antes, poblacion_despues):
    etiqueta_prevista = (df_pob["tickets_soporte"] >= 3).astype(int)
    regla_es_la_misma &= etiqueta_prevista.equals(df_pob["abandono"])

print("Media de tickets_soporte antes:", poblacion_antes["tickets_soporte"].mean(),
      "-- después:", poblacion_despues["tickets_soporte"].mean())
print("¿La regla 'tickets_soporte >= 3 => abandono' sigue explicando la etiqueta en ambas poblaciones?", regla_es_la_misma)
print("Esto es DATA DRIFT: cambió P(X) (la media de tickets_soporte pasó de 1,0 a 3,75), no P(Y|X).")

print()

# Ejemplo B -- CONCEPT DRIFT: la distribución de entrada NO cambia, la relación con la etiqueta SÍ.
# Imaginemos que la empresa introduce un reembolso automático tras el tercer ticket: la MISMA cantidad de tickets ya no predice abandono igual.
poblacion_antes_politica = pd.DataFrame({"tickets_soporte": [3, 4, 3, 4], "abandono": [1, 1, 1, 1]})
poblacion_despues_politica = pd.DataFrame({"tickets_soporte": [3, 4, 3, 4], "abandono": [0, 0, 0, 1]})

misma_distribucion_x = poblacion_antes_politica["tickets_soporte"].tolist() == poblacion_despues_politica["tickets_soporte"].tolist()
tasa_abandono_antes = poblacion_antes_politica["abandono"].mean()
tasa_abandono_despues = poblacion_despues_politica["abandono"].mean()

print("¿Los valores de tickets_soporte son idénticos antes y después?", misma_distribucion_x)
print("Tasa de abandono entre clientes con >=3 tickets, antes:", tasa_abandono_antes,
      "-- después de la nueva política de reembolso:", tasa_abandono_despues)
print("Esto es CONCEPT DRIFT: P(X) no cambió (mismos tickets_soporte), pero P(Y|X) sí (de 100% a 25% de abandono con >=3 tickets).")

assert misma_distribucion_x and tasa_abandono_antes == 1.0 and tasa_abandono_despues == 0.25

**¿Qué observas?** En el ejemplo A la media de `tickets_soporte` pasa de 1,0 a 3,75 -- P(X) cambió claramente -- pero la misma regla ("3 o más tickets implica abandono") sigue prediciendo la etiqueta exacta en las dos poblaciones: es *data drift* puro. En el ejemplo B, los valores de `tickets_soporte` son idénticos en las dos poblaciones -- P(X) no cambió en absoluto -- pero la fracción de clientes que abandona con 3 o más tickets cae del 100 % al 25 % tras el cambio de política: es *concept drift* puro. En un caso real casi nunca son tan limpios ni tan separables; el punto de este ejemplo es solo fijar la diferencia conceptual antes de medirla con datos reales.

---
## 2. El índice de estabilidad poblacional (PSI), a mano

Antes de aplicarlo a datos reales, un ejemplo mínimo verificable a mano: 4 buckets, con un porcentaje de referencia y un porcentaje nuevo ya calculados.

In [ ]:
# Ejemplo mínimo de PSI, verificable a mano, con 4 buckets y porcentajes ya calculados (no cuentas brutas).
porcentaje_referencia = np.array([0.40, 0.30, 0.20, 0.10])
porcentaje_nuevo = np.array([0.10, 0.20, 0.30, 0.40])


def calcular_psi(porcentaje_referencia, porcentaje_nuevo):
    """PSI = suma, en cada bucket, de (nuevo% - referencia%) * ln(nuevo% / referencia%)."""
    return float(np.sum((porcentaje_nuevo - porcentaje_referencia) * np.log(porcentaje_nuevo / porcentaje_referencia)))


psi_ejemplo = calcular_psi(porcentaje_referencia, porcentaje_nuevo)
print("PSI del ejemplo (4 buckets):", round(psi_ejemplo, 5))

# Verificación a mano, bucket a bucket: (nuevo-ref)*ln(nuevo/ref)
aportes_a_mano = [
    (0.10 - 0.40) * np.log(0.10 / 0.40),  # ≈0,41589
    (0.20 - 0.30) * np.log(0.20 / 0.30),  # ≈0,04055
    (0.30 - 0.20) * np.log(0.30 / 0.20),  # ≈0,04055
    (0.40 - 0.10) * np.log(0.40 / 0.10),  # ≈0,41589
]
print("Aportes por bucket (a mano):", [round(a, 5) for a in aportes_a_mano])
print("Suma de los aportes:", round(sum(aportes_a_mano), 5))
assert abs(sum(aportes_a_mano) - psi_ejemplo) < 1e-9, "la función y el cálculo a mano deben coincidir"
assert psi_ejemplo > 0.25, "este ejemplo se diseñó para caer en la zona de drift severo (PSI>0,25)"
print("PSI > 0,25: con la convención habitual, este desplazamiento se clasificaría como drift SEVERO.")

---
## 3. PSI y Kolmogorov-Smirnov reales: `churn.csv` frente a `churn_v2.csv`

Aplicamos `calcular_psi` (envuelto en `psi_numerico`/`psi_categorico`, que construyen los buckets) y `scipy.stats.ks_2samp` a 5 columnas: 3 que se desplazaron a propósito al generar `churn_v2.csv` (`uso_mensual_horas`, `tickets_soporte`, `contrato`) y 2 columnas de control que NO se tocaron (`antiguedad_meses`, `factura_mensual`).

> **Atención:** los buckets del PSI se fijan siempre con los cuantiles de la REFERENCIA (`churn.csv`), nunca con los del dato nuevo -- si se recalcularan sobre el dato nuevo, el PSI podría salir artificialmente bajo incluso con un desplazamiento real.

In [ ]:
def psi_numerico(referencia, nuevo, buckets=10):
    """PSI para una columna numérica: los bordes de los buckets se fijan con los cuantiles de la REFERENCIA (nunca del nuevo dato), y se reutilizan tal cual para contar el nuevo."""
    bordes = np.unique(np.quantile(referencia, np.linspace(0, 1, buckets + 1)))
    bordes[0], bordes[-1] = -np.inf, np.inf
    cuenta_ref, _ = np.histogram(referencia, bins=bordes)
    cuenta_nuevo, _ = np.histogram(nuevo, bins=bordes)
    porc_ref = np.clip(cuenta_ref / len(referencia), 1e-4, None)
    porc_nuevo = np.clip(cuenta_nuevo / len(nuevo), 1e-4, None)
    return calcular_psi(porc_ref, porc_nuevo)


def psi_categorico(referencia, nuevo):
    """PSI para una columna categórica: cada categoría es un bucket."""
    categorias = sorted(set(referencia.unique()) | set(nuevo.unique()))
    porc_ref = np.array([np.clip((referencia == c).mean(), 1e-4, None) for c in categorias])
    porc_nuevo = np.array([np.clip((nuevo == c).mean(), 1e-4, None) for c in categorias])
    return calcular_psi(porc_ref, porc_nuevo)


COLUMNAS_A_COMPROBAR = ["uso_mensual_horas", "tickets_soporte", "contrato", "antiguedad_meses", "factura_mensual"]
DESPLAZADAS_A_PROPOSITO = {"uso_mensual_horas", "tickets_soporte", "contrato"}

resultados = []
for columna in COLUMNAS_A_COMPROBAR:
    if columna == "contrato":
        psi = psi_categorico(df_v1[columna], df_v2[columna])
        ks_stat, ks_p = None, None
    else:
        psi = psi_numerico(df_v1[columna].astype(float), df_v2[columna].astype(float))
        ks_res = ks_2samp(df_v1[columna], df_v2[columna])
        ks_stat, ks_p = float(ks_res.statistic), float(ks_res.pvalue)
    resultados.append({
        "columna": columna,
        "desplazada_a_proposito": columna in DESPLAZADAS_A_PROPOSITO,
        "psi": round(psi, 4),
        "ks_estadistico": None if ks_stat is None else round(ks_stat, 4),
        "ks_p_valor": ks_p,
    })

tabla_drift = pd.DataFrame(resultados)
pd.set_option("display.width", 140)
print(tabla_drift.to_string(index=False))

---
## 4. Interpretar el PSI con una convención estándar

In [ ]:
def interpretar_psi(psi):
    """Convención estándar (Siddiqi, "Credit Risk Scorecards"): <0,1 sin drift; 0,1-0,25 drift moderado; >0,25 drift severo."""
    if psi < 0.10:
        return "sin drift"
    if psi < 0.25:
        return "drift moderado"
    return "drift severo"


tabla_drift["interpretacion"] = tabla_drift["psi"].apply(interpretar_psi)
print(tabla_drift[["columna", "desplazada_a_proposito", "psi", "interpretacion"]].to_string(index=False))

for _, fila in tabla_drift.iterrows():
    if fila["desplazada_a_proposito"]:
        assert fila["psi"] > 0.10, "una columna desplazada a propósito debería mostrar al menos drift moderado"
    else:
        assert fila["psi"] < 0.10, "una columna de control no debería mostrar drift"
print()
print("Las 3 columnas desplazadas a propósito muestran drift moderado o severo; las 2 columnas de control (no tocadas al generar churn_v2.csv) no muestran ningún drift. PSI y KS distinguen correctamente el desplazamiento real del ruido de muestreo.")

---
## 5. La misma conclusión con Evidently

`Report([DataDriftPreset()])` compara automáticamente cada columna con el método adecuado a su tipo (distancia de Wasserstein normalizada para numéricas, distancia de Jensen-Shannon para categóricas) y un umbral por defecto de 0,10.

In [ ]:
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset

conjunto_referencia = Dataset.from_pandas(df_v1[COLUMNAS_ENTRADA], data_definition=DataDefinition())
conjunto_nuevo = Dataset.from_pandas(df_v2[COLUMNAS_ENTRADA], data_definition=DataDefinition())

informe = Report([DataDriftPreset()])
resultado = informe.run(reference_data=conjunto_referencia, current_data=conjunto_nuevo)
resultado_dict = resultado.dict()

RUTA_INFORME = os.path.join(CARPETA, "informe_drift_evidently.json")
with open(RUTA_INFORME, "w", encoding="utf-8") as f:
    json.dump({"metrics": [{"nombre": m["metric_name"], "valor": (m["value"] if isinstance(m["value"], dict) else float(m["value"]))} for m in resultado_dict["metrics"]]}, f, ensure_ascii=False, indent=2, default=float)

filas_evidently = []
for m in resultado_dict["metrics"]:
    if m["metric_name"].startswith("DriftedColumnsCount"):
        continue
    columna = m["config"]["column"]
    valor = float(m["value"])
    filas_evidently.append({"columna": columna, "metodo": m["config"]["method"], "distancia": round(valor, 4),
                             "supera_umbral_0_10": valor > 0.10})

tabla_evidently = pd.DataFrame(filas_evidently).sort_values("columna").reset_index(drop=True)
print(tabla_evidently.to_string(index=False))

conteo_drift = next(m["value"] for m in resultado_dict["metrics"] if m["metric_name"].startswith("DriftedColumnsCount"))
print()
print("Columnas marcadas con drift por Evidently:", int(conteo_drift["count"]), "de 7 columnas de entrada (", round(conteo_drift["share"] * 100, 1), "%).")

columnas_marcadas = set(tabla_evidently.loc[tabla_evidently["supera_umbral_0_10"], "columna"])
assert columnas_marcadas == DESPLAZADAS_A_PROPOSITO, "Evidently debería marcar exactamente las 3 columnas desplazadas a propósito"
print("Evidently, con su propio método por columna (Wasserstein para numéricas, Jensen-Shannon para categóricas), marca EXACTAMENTE las mismas 3 columnas que nuestro PSI/KS manual -- dos formas de medir independientes, misma conclusión.")

---
## 6. Reentrenar sobre el dato con *drift* y medir el impacto real

Reentrenamos la misma Pipeline (regresión logística, `class_weight="balanced"`, `C=1,0`) pero ahora sobre `churn_v2.csv`, y la evaluamos en el MISMO conjunto de prueba de `churn.csv` que usan las clases 4, 5, 7 y 10 -- así la comparación es limpia: mismos datos de prueba, solo cambia con qué se entrenó.

In [ ]:
def construir_pipeline():
    preprocesador = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
        ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
    ])
    clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
    return Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])


# El conjunto de prueba de churn.csv es el mismo de las clases 4, 5, 7 y 10 (misma semilla, mismo test_size): es el punto de comparación fijo de todo este capítulo.
X_v1 = df_v1[COLUMNAS_ENTRADA]
y_v1 = df_v1["abandono"]
X_train, X_test, y_train, y_test = train_test_split(X_v1, y_v1, test_size=0.25, random_state=SEMILLA, stratify=y_v1)

modelo_original = construir_pipeline()
modelo_original.fit(X_train, y_train)
f1_original = f1_score(y_test, modelo_original.predict(X_test))
auc_original = roc_auc_score(y_test, modelo_original.predict_proba(X_test)[:, 1])

modelo_reentrenado = construir_pipeline()
modelo_reentrenado.fit(df_v2[COLUMNAS_ENTRADA], df_v2["abandono"])
f1_reentrenado = f1_score(y_test, modelo_reentrenado.predict(X_test))
auc_reentrenado = roc_auc_score(y_test, modelo_reentrenado.predict_proba(X_test)[:, 1])

joblib.dump(modelo_reentrenado, os.path.join(CARPETA, "modelo_reentrenado.pkl"))

print("Modelo entrenado en churn.csv, evaluado en el TEST de churn.csv (línea base, clases 4/5/7/10):")
print("  F1:", round(f1_original, 4), " AUC:", round(auc_original, 4))
print("Modelo REENTRENADO en churn_v2.csv (con el drift), evaluado en el MISMO test de churn.csv:")
print("  F1:", round(f1_reentrenado, 4), " AUC:", round(auc_reentrenado, 4))
print("Diferencia de F1:", round(f1_reentrenado - f1_original, 4))

---
## 7. La misma pregunta de la clase 6: ¿qué esconde la media?

In [ ]:
# La diferencia global de F1 es pequeña -- exactamente el mismo riesgo que la clase 6: una métrica agregada puede esconder un problema real en un subgrupo. Repetimos el análisis por segmento de "plan".
X_test_con_etiqueta = X_test.copy()
X_test_con_etiqueta["abandono_real"] = y_test.values

filas_segmento = []
for segmento in sorted(df_v1["plan"].unique()):
    mascara = X_test_con_etiqueta["plan"] == segmento
    y_segmento = X_test_con_etiqueta.loc[mascara, "abandono_real"]
    X_segmento = X_test_con_etiqueta.loc[mascara, COLUMNAS_ENTRADA]
    f1_orig_seg = f1_score(y_segmento, modelo_original.predict(X_segmento))
    f1_reen_seg = f1_score(y_segmento, modelo_reentrenado.predict(X_segmento))
    filas_segmento.append({"plan": segmento, "n_clientes": int(mascara.sum()),
                            "F1_modelo_original": round(f1_orig_seg, 4),
                            "F1_modelo_reentrenado": round(f1_reen_seg, 4)})

tabla_segmentos = pd.DataFrame(filas_segmento)
print(tabla_segmentos.to_string(index=False))

peor_caso = tabla_segmentos.loc[tabla_segmentos["F1_modelo_reentrenado"].idxmin()]
assert peor_caso["plan"] == "premium" and peor_caso["F1_modelo_reentrenado"] < peor_caso["F1_modelo_original"], "el segmento premium debería empeorar claramente tras reentrenar con el dato desplazado"
print()
print("El segmento", peor_caso["plan"], "pasa de F1", peor_caso["F1_modelo_original"],
      "a F1", peor_caso["F1_modelo_reentrenado"],
      "tras reentrenar: el modelo reentrenado deja de detectar CUALQUIER abandono real en ese segmento, aunque el F1 global apenas cambie. Es un impacto real, no solo una diferencia estadística en las columnas de entrada.")

---
## 8. Disparador de reentrenamiento

Un disparador explícito, con dos condiciones independientes: `PSI > 0,25` en al menos una columna, O el F1 medido cae por debajo de un umbral mínimo.

In [ ]:
def evaluar_disparador(tabla_drift, f1_actual, umbral_psi=0.25, umbral_f1=0.25):
    """Disparador explícito: reentrenar si (a) al menos una columna clave supera PSI>umbral_psi, O (b) el F1 medido cae por debajo de umbral_f1. Devuelve qué condición se cumplió."""
    columnas_con_drift_severo = tabla_drift.loc[tabla_drift["psi"] > umbral_psi, "columna"].tolist()
    disparado_por_drift = len(columnas_con_drift_severo) > 0
    disparado_por_calidad = f1_actual < umbral_f1
    return {
        "disparar_reentrenamiento": disparado_por_drift or disparado_por_calidad,
        "disparado_por_drift": disparado_por_drift,
        "columnas_con_drift_severo": columnas_con_drift_severo,
        "disparado_por_calidad": disparado_por_calidad,
    }


decision = evaluar_disparador(tabla_drift, f1_original)
print("Evaluación del disparador (con las cifras de este mismo notebook):")
for clave, valor in decision.items():
    print(" ", clave, ":", valor)

assert decision["disparar_reentrenamiento"] and decision["disparado_por_drift"] and not decision["disparado_por_calidad"], \
    "en este ejemplo el disparador debería activarse por drift, no por caída del F1 global"
print()
print("El disparador se activa por DRIFT (PSI>0,25 en", decision["columnas_con_drift_severo"], "), NO por caída del F1 global -- que de hecho no cayó. Si solo vigiláramos el F1 agregado, como advertía la clase 10, este problema pasaría desapercibido hasta que un subgrupo entero (plan premium) dejara de detectarse.")

---
## 9. El riesgo de reentrenar con datos contaminados

Este capítulo reentrenó asumiendo que `churn_v2.csv` refleja un cambio real y legítimo en el comportamiento de los clientes (más contratos mensuales, menos uso, más tickets). Pero un desplazamiento con esta misma forma estadística también podría venir de un **bug de captura de datos** -- por ejemplo, un fallo en el sistema de facturación que subcuenta `uso_mensual_horas` para los contratos mensuales, o un cambio no anunciado en cómo se clasifican los tickets. PSI y KS no distinguen esas dos causas: solo dicen que P(X) cambió, no POR QUÉ.

> **Atención:** reentrenar automáticamente en cuanto se dispara el umbral, sin investigar la causa, arriesga perpetuar el problema: si el drift viene de un bug, el modelo reentrenado aprende una relación contaminada por ese bug, y cada reentrenamiento posterior sobre el mismo flujo de datos defectuoso lo reafirma en vez de corregirlo. El disparador de la sección 8 dice CUÁNDO investigar; no reemplaza la investigación -- solo un ingeniero de datos o quien opere la fuente puede confirmar si el cambio es real o un defecto.

---
## 10. Cerrando el ciclo del curso

Si el equipo confirma que el drift es real (no un bug) y decide reentrenar, el modelo reentrenado no se despliega directamente: vuelve a pasar por el mismo ciclo que ya construimos en este curso -- **versionado** (clase 3: una nueva versión de los datos y del modelo, con su propio hash), **seguimiento** (clase 4: un nuevo experimento registrado en MLflow, comparable con los anteriores), **pruebas** (clase 6: la misma suite de `pytest` y `pandera` debe volver a pasar, incluida la comprobación de F1 mínimo) y **despliegue** (clase 9: canary o shadow antes de sustituir el modelo en producción, nunca un reemplazo directo). El *drift* no es un evento aislado: es la señal que reinicia el mismo ciclo, no un atajo para saltárselo.

---
## 11. Resumen y cierre

- Distinguimos *data drift* (cambia P(X)) de *concept drift* (cambia P(Y|X)) con un ejemplo pequeño de cada uno.
- Implementamos PSI a mano (con un ejemplo verificable) y usamos KS de `scipy`, sobre un cambio real entre `churn.csv` y `churn_v2.csv`: las 3 columnas desplazadas a propósito muestran drift, las 2 de control no.
- Confirmamos la misma conclusión con Evidently, con métodos distintos por tipo de columna.
- Reentrenamos sobre el dato con drift y medimos un impacto real, no solo estadístico: el F1 global casi no cambia, pero un segmento entero (plan premium) deja de detectarse.
- Definimos un disparador explícito (PSI o F1) y discutimos el riesgo de reentrenar sobre datos ya contaminados por el problema que causó el drift.
- Cerramos el ciclo: el modelo reentrenado vuelve a pasar por versionado (clase 3), seguimiento (clase 4), pruebas (clase 6) y despliegue (clase 9).

*Continúa con la clase 12: Gobernanza, costes y LLMOps + proyecto.*

---
## Limpieza final

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)